# 深度學習框架應用（大學部）｜主題 4（上）：從一個神經元到一整個網路

這本筆記本配合投影片《主題 4（上）：全連接神經網路 MLP（多層感知器）》。每一題都用你自己算過的東西當例子：學期成績（配分×分數再加起來）、學測科系加權、答錯倒扣、每一大題最低 0 分、樓梯間的兩個開關、班長選舉的得票率。
算法跟生活裡一模一樣，差別只在神經網路的配分（權重）是電腦看很多有答案的例子自己調出來的。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 學期成績就是一個神經元（加權、偏置、ReLU、nn.Linear） | 第 92 頁 | 第 15、18、24 頁 |
| 二 | 兩個科系，一次算完（一層＝矩陣乘法、形狀規則） | 第 93 頁 | 第 28、29、30 頁 |
| 三 | 樓梯間的兩個開關 XOR：一層 vs 兩層 | 第 94 頁 | 第 33、36、37 頁 |
| 四 | 加權再加權，還是一次加權＋數配分（參數量） | 第 95 頁 | 第 40、41、55 頁 |
| 五 | 得票率、扣分、往回調（softmax、交叉熵、梯度） | 第 96 頁 | 第 62、67、78 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W04上_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」，再由上往下做。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 五題都是很小的張量，**CPU 就跑得動，不用開 GPU**；練習三要讓電腦自己練 1,000 次，大約幾秒鐘。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**
5. 文字題裡的 **N 是你自己學號的末兩碼**（例如學號尾數 07，N＝07），每個人算出來的數字都不一樣。

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區，期限照 Flipclass 規定。

## 環境準備

五題都只用到 PyTorch，先跑這一格。最後一行讓 PyTorch 印數字時只印到小數第 3 位，比較好對答案。

In [ ]:
# ===== 環境準備 =====
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.set_printoptions(precision=3, sci_mode=False)  # 印到小數第 3 位
print('PyTorch：', torch.__version__)

## 練習一：學期成績就是一個神經元 (20%)

> 題目在投影片**第 92 頁**｜觀念在第 15、18、24 頁

**生活例子**：作業 80、期中 70、期末 90，配分 30%、30%、40%，再加平時加分 2 分。另外一大題答對 +2、答錯 −1，小華答對 1 題、答錯 4 題。

**這題在練**：一個神經元的算法「每一項乘上配分、加起來、再加分」，再用 `nn.Linear(3, 1)` 設成同樣的配分對答案；最後用 ReLU 做出「這一大題最低 0 分」

**步驟**

1. 填 `.sum()`：每一項乘上配分，全部加起來，再加平時加分，學期成績應該是 83
2. 把配分 w 抄進 `grade.weight`，讓 `nn.Linear` 算一次，也應該是 83
3. 倒扣那一大題算出 −2：填 ReLU，看它會不會變成 0

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習一：學期成績就是一個神經元
x = torch.tensor([80., 70., 90.])  # 作業、期中、期末
w = torch.tensor([0.3, 0.3, 0.4])  # 配分（權重）
b = 2.                             # 平時加分（偏置）
y = (x * w).___() + b              # ✏️ 乘完全部加起來
print('學期成績', round(y.item(), 2))
grade = nn.Linear(3, 1)            # 一個神經元
with torch.no_grad():
    grade.weight[:] = ___          # ✏️ 抄配分
    grade.bias[:] = b              # 抄加分
print('nn.Linear', round(grade(x).item(), 2))
z2 = torch.tensor([1., 4.]) @ torch.tensor([2., -1.])
r2 = torch.___(z2)                 # ✏️ 最低 0 分
print('倒扣', z2.item(), '→', r2.item())  # 對 1 錯 4

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行結果（學期成績、nn.Linear、倒扣 → 最低 0 分）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 期末考的配分 0.4 比作業的 0.3 大，代表什麼？倒扣那一大題的權重 −1 又代表什麼？**

（在這裡寫）

**2. 用你的學號末兩碼 N 當作業分數（期中 70、期末 90、平時加分 2 不變），學期成績是多少？照「60 分以上輸出 1（及格）」的規則，輸出是 1 還是 0？**

（在這裡寫）

## 練習二：兩個科系，一次算完 (20%)

> 題目在投影片**第 93 頁**｜觀念在第 28、29、30 頁

**生活例子**：同一份學測成績，資工系加權國×1、英×1、數×2，外文系加權國×1、英×2、數×1。同學甲的國英數是 12、13、10 級分，同學乙是 14、10、15 級分。

**這題在練**：一層＝很多套配分同時算同一份資料＝一次矩陣乘法：把兩個科系的加權疊成 W（2×3），兩位同學疊成 X（2×3），用 `X @ W.T` 一次算完，並看懂形狀規則「內側相等、取外側」

**步驟**

1. 填 `W.T`：加權表轉置成 (3, 2)，同學的 3 科才對得上加權表的 3 科
2. 看印出來的形狀 (2, 3) @ (3, 2) → (2, 2)，以及 2 位同學×2 個科系的加權分
3. 用 `nn.Linear(3, 2, bias=False)` 設同樣的加權，結果應該一模一樣；注意它的權重形狀是 (2, 3)

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習二：兩個科系，一次算完
W = torch.tensor([[1., 1., 2.],       # 資工系的加權
                  [1., 2., 1.]])      # 外文系的加權
X = torch.tensor([[12., 13., 10.],    # 同學甲：國英數
                  [14., 10., 15.]])   # 同學乙
Z = X @ W.___                         # ✏️ 加權表轉置
print(X.shape, '@', W.T.shape, '→', Z.shape)
print('加權分：', Z.tolist())
dept = nn.Linear(3, ___, bias=False)  # ✏️ 幾個科系
with torch.no_grad():
    dept.weight[:] = ___              # ✏️ 抄加權
print('nn.Linear', dept(X).tolist())
print('權重形狀', dept.weight.shape)  # (輸出, 輸入)

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：形狀那一行、加權分、nn.Linear 的結果和權重形狀

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 為什麼要寫 W.T？用「內側相等、取外側」說明 (2, 3) @ (3, 2) → (2, 2)，結果的每一格代表什麼。**

（在這裡寫）

**2. 同學丙的國文 12、英文 13 級分，數學級分是「你學號末兩碼 N 的個位數＋6」。資工系和外文系的加權分各是多少？**

（在這裡寫）

## 練習三：樓梯間的兩個開關（XOR） (20%)

> 題目在投影片**第 94 頁**｜觀念在第 33、36、37 頁

**生活例子**：樓梯間樓上、樓下各有一個開關。規則：只有一個開關扳上去時燈亮；兩個都沒扳、兩個都扳上去，燈都是暗的。

**這題在練**：先用手設的配分解開 XOR（h₁＝扳上去幾個、h₂＝兩個都扳上去了嗎、燈＝h₁−2h₂）；再讓電腦自己練「一層」和「兩層（中間 16 個神經元）」，比一比

**步驟**

1. 先執行「準備」那一格：放好四種開關情況 X、燈亮不亮 y，以及讓電腦自己練的 `train_xor()`
2. 填 h₂ 要減的數和燈的算法，手設配分應該得到 [0, 1, 1, 0]
3. 填中間那一層的神經元數 16，看「一層」最多只到 75%、「兩層」到 100%

> `train_xor()` 用的是主題 4（下）才會細講的寫法：BCEWithLogitsLoss 算扣分、Adam 調配分、練 1,000 次。這題只要看結果。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改）=====
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])  # (樓下, 樓上)：1＝扳上去
y = torch.tensor([[0.], [1.], [1.], [0.]])  # 只扳一個，燈才亮


def train_xor(model, steps=1000, lr=0.05):
    """讓電腦自己調配分：用 Adam 練 steps 次，回傳 (準確率, 扣分)。"""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()      # 亮或不亮：二分類的扣分方法
    for _ in range(steps):
        loss = loss_fn(model(X), y)       # 四種情況都看一次、算扣分
        opt.zero_grad()                   # 清掉上一次算的責任
        loss.backward()                   # 往回算每個配分的責任
        opt.step()                        # 照責任調配分
    with torch.no_grad():
        out = model(X)
        acc = ((out > 0).float() == y).float().mean().item()
        return acc, loss_fn(out, y).item()


print('兩個開關的四種情況：', X.tolist())
print('燈亮不亮：', y.flatten().tolist())

In [ ]:
# ✏️ 練習三：樓梯間的兩個開關（XOR）
s = X[:, 0] + X[:, 1]         # 扳上去幾個
h1 = torch.relu(s)
h2 = torch.relu(s - ___)      # ✏️ 兩個都扳上去了嗎
light = h1 - ___ * h2         # ✏️ 燈亮不亮
print('手設配分：', light.int().tolist())
torch.manual_seed(0)          # 固定亂數
one = nn.Linear(2, 1)         # 一層：一條直線
two = nn.Sequential(nn.Linear(2, ___), nn.ReLU(),  # ✏️
                    nn.Linear(16, 1))  # 兩層
for name, m in [('一層', one), ('兩層', two)]:
    acc, loss = train_xor(m)  # 電腦自己調配分
    print(f'{name}：準確率 {acc:.0%}，扣分 {loss:.3f}')

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：手設配分的結果、一層和兩層的準確率與扣分

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 為什麼「一層」怎麼練都到不了 100%，扣分停在 0.693 附近？**

（在這裡寫）

**2. 把 torch.manual_seed(0) 改成 torch.manual_seed(N)（N＝學號末兩碼）再跑一次，寫下兩種的準確率和扣分。為什麼中間放 16 個神經元，不只放手設的 2 個？**

（在這裡寫）

## 練習四：加權再加權＋數配分 (20%)

> 題目在投影片**第 95 頁**｜觀念在第 40、41、55 頁

**生活例子**：先把國英數算成「文科分＝國＋英」「理科分＝數」，再算科系分＝文科×1＋理科×2；另外看一大題「最低 0 分」這種規則夾在中間會怎樣。

**這題在練**：親眼看到「中間沒有規則」時兩層加權可以併成一套配分 [1, 1, 2]，中間加上 ReLU（最低 0 分）就併不起來；再用 `sum(p.numel())` 數 784 → 128 → 10 一共有幾格配分

**步驟**

1. 填 `W1`：兩套配分相乘 W2 @ W1，併成一套 [1, 1, 2]，兩階段和一次算都是 45
2. 換一組數字，中間那一層是 [−1, 1]：填 ReLU，看「最低 0 分」得 0、沒有規則得 −1
3. 填最後一層的 10 和 `numel`，參數量應該是 101,770

> 第二部分的數字跟投影片「數學小工具：沒有激活，兩層併成一層」那頁一樣：第一層 [[1, 2], [0, 1]]、加 [1, 0]，第二層兩個加起來再 −1。

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習四：加權再加權＋數配分
x = torch.tensor([12., 13., 10.])       # 國英數
W1 = torch.tensor([[1., 1., 0.], [0., 0., 1.]])  # 文、理
W2 = torch.tensor([1., 2.])             # 文×1＋理×2
print('兩階段', (W2 @ (W1 @ x)).item())
Wm = W2 @ ___                           # ✏️ 併成一套配分
print('一次', (Wm @ x).item(), Wm.tolist())
A = torch.tensor([[1., 2.], [0., 1.]])  # 換一組數字
h = A @ torch.tensor([-4., 1.]) + torch.tensor([1., 0.])
print('沒有規則', h.sum().item() - 1)   # 中間照算
print('最低 0 分', torch.___(h).sum().item() - 1)  # ✏️
m = nn.Sequential(nn.Linear(784, 128), nn.ReLU(),
                  nn.Linear(128, ___))  # ✏️ 0～9
print(sum(p.___() for p in m.parameters()))  # ✏️ 參數量

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：兩階段與一次、併成的配分、沒有規則與最低 0 分、參數量

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 用國英數的例子說明：為什麼中間沒有「最低 0 分」這種規則，兩層加權就等於一次加權？加上 ReLU 以後為什麼併不起來？**

（在這裡寫）

**2. 用你的學號末兩碼 N：把網路改成 784 → (100＋N) → 10，手算參數量（寫出算式），再改程式驗證。**

（在這裡寫）

## 練習五：得票率、扣分、往回調 (20%)

> 題目在投影片**第 96 頁**｜觀念在第 62、67、78 頁

**生活例子**：三個候選答案的分數是 [2, 1, 0]，正確答案是第 0 個。先把分數換成得票率，看正確答案拿到多少票來扣分，再往回算責任、照學習率 0.5 調一次。

**這題在練**：自己寫 softmax（得票率）和交叉熵（扣分），跟 `F.cross_entropy` 對答案；再用 `backward()` 看梯度，照學習率 0.5 調一次，扣分會變小

**步驟**

1. 填 `.sum()` 做出 softmax，得票率應該是 [0.665, 0.245, 0.090]
2. 填正確答案 t，算出扣分 0.408，和 `F.cross_entropy` 一樣
3. 填 `backward` 和學習率 0.5：梯度是 [−0.335, 0.245, 0.090]，調一次後扣分變 0.326

> 這題為了好算，直接調三個分數 z；實際訓練時調的是配分（權重），做法一樣是「減掉學習率×梯度」。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習五：得票率、扣分、往回調
z = torch.tensor([2., 1., 0.], requires_grad=True)
t = torch.tensor(0)                    # 正確答案：第 0 個
p = torch.exp(z) / torch.exp(z).___()  # ✏️ 得票率
print('得票率：', p.detach())
loss = -torch.log(p[___])              # ✏️ 看正確答案
print('手算扣分', round(loss.item(), 3))
ce = F.cross_entropy(z, t).item()      # PyTorch 的扣分
print('F.cross_entropy', round(ce, 3))
loss.___()                             # ✏️ 往回算責任
print('梯度：', z.grad)
with torch.no_grad():
    z -= ___ * z.grad                  # ✏️ 學習率 0.5
print('調一次後', round(F.cross_entropy(z, t).item(), 3))

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：得票率、手算扣分、F.cross_entropy、梯度、調一次後的扣分

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 梯度 [−0.335, 0.245, 0.090] 和得票率有什麼關係？為什麼第 0 個是負的？**

（在這裡寫）

**2. 用你的學號末兩碼 N：分數改成 [2, 1, N÷10]，答案還是第 0 個。扣分變大還是變小？用得票率說明，並寫出你的扣分。**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：W.T 是什麼意思） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W04上_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **文字題**：五題的 ✏️ 文字回答都寫好，用到 N 的地方換成自己的學號末兩碼。
5. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
6. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
7. **下載**：「檔案 › 下載 › 下載 .ipynb」。
8. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。